In [3]:
from pathlib import Path

import pandas as pd

In [4]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

In [5]:
RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

print("Raw directory exists:", RAW_DATA_DIR.exists())
print("Processed directory exists:", PROCESSED_DATA_DIR.exists())

Raw directory exists: True
Processed directory exists: True


In [6]:
# file map
dataset_files = {
    "customers": RAW_DATA_DIR / "olist_customers_dataset.csv",
    "geolocation": RAW_DATA_DIR / "olist_geolocation_dataset.csv",
    "order_items": RAW_DATA_DIR / "olist_order_items_dataset.csv",
    "payments": RAW_DATA_DIR / "olist_order_payments_dataset.csv",
    "reviews": RAW_DATA_DIR / "olist_order_reviews_dataset.csv",
    "orders": RAW_DATA_DIR / "olist_orders_dataset.csv",
    "products": RAW_DATA_DIR / "olist_products_dataset.csv",
    "sellers": RAW_DATA_DIR / "olist_sellers_dataset.csv",
    "category_translation": RAW_DATA_DIR / "product_category_name_translation.csv",
}

In [7]:
raw_datasets = {
    dataset_name: pd.read_csv(file_path)
    for dataset_name, file_path in dataset_files.items()
}

for dataset_name, df in raw_datasets.items():
    print(f"{dataset_name:<22} {df.shape}")

customers              (99441, 5)
geolocation            (1000163, 5)
order_items            (112650, 7)
payments               (103886, 5)
reviews                (99224, 7)
orders                 (99441, 8)
products               (32951, 9)
sellers                (3095, 4)
category_translation   (71, 2)


# 1. Customers dataset

In [6]:
customers_raw = raw_datasets["customers"]

customers_clean = customers_raw.copy()

In [7]:
customers_clean.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [8]:
customers_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [9]:
# Converting zip code from int to string
customers_clean["customer_zip_code_prefix"] = (
    customers_clean["customer_zip_code_prefix"]
    .astype("string")
    .str.zfill(5) # adds zeroes to begining
)

In [10]:
customers_clean["customer_zip_code_prefix"].head()

0    14409
1    09790
2    01151
3    08775
4    13056
Name: customer_zip_code_prefix, dtype: string

In [11]:
# Standardize text columns
text_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
]

for column in text_columns:
    customers_clean[column] = (
        customers_clean[column]
        .astype("string")
        .str.strip() # remove accidental spaces
    )

In [12]:
customers_clean["customer_state"] = (
    customers_clean["customer_state"]
    .str.upper()
)

In [13]:
# Missing values (no missing values found)
customers_clean.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [14]:
# Checking Duplicate records
print("Exact duplicate rows:", customers_clean.duplicated().sum())
print("Duplicate customer_id:", customers_clean["customer_id"].duplicated().sum())
print(
    "Repeated customer_unique_id:",
    customers_clean["customer_unique_id"].duplicated().sum(),
)

Exact duplicate rows: 0
Duplicate customer_id: 0
Repeated customer_unique_id: 3345


From discovery:

- No exact duplicates
- customer_id is unique
- customer_unique_id can repeat validly

In [15]:
# Validate state codes
customer_states = sorted(
    customers_clean["customer_state"].dropna().unique()
)

print("Number of states:", len(customer_states))
print(customer_states)

Number of states: 27
['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']


In [16]:
# Validate city values : check for empty strings
print(
    "Empty customer_city values:",
    customers_clean["customer_city"].eq("").sum(),
)

print(
    "Empty customer_state values:",
    customers_clean["customer_state"].eq("").sum(),
)

Empty customer_city values: 0
Empty customer_state values: 0


In [17]:
# final validation
print("Original shape:", customers_raw.shape)
print("Cleaned shape:", customers_clean.shape)

print(
    "Unique customer_id:",
    customers_clean["customer_id"].nunique(),
)

print(
    "Unique customer_unique_id:",
    customers_clean["customer_unique_id"].nunique(),
)

print(
    "Total missing values:",
    customers_clean.isna().sum().sum(),
)

print(
    "Exact duplicates:",
    customers_clean.duplicated().sum(),
)

Original shape: (99441, 5)
Cleaned shape: (99441, 5)
Unique customer_id: 99441
Unique customer_unique_id: 96096
Total missing values: 0
Exact duplicates: 0


In [18]:
# comparing with the raw (before , after)
dtype_comparison = pd.DataFrame(
    {
        "raw_dtype": customers_raw.dtypes.astype(str),
        "clean_dtype": customers_clean.dtypes.astype(str),
    }
)

dtype_comparison

,raw_dtype,clean_dtype
customer_id,str,string
customer_unique_id,str,string
customer_zip_code_prefix,int64,string
customer_city,str,string
customer_state,str,string


In [19]:
# saving the cleaned customers dataset
customers_output_path = (
    PROCESSED_DATA_DIR / "customers_clean.csv"
)

customers_clean.to_csv(
    customers_output_path,
    index=False,
)

print(f"Saved cleaned customers data to: {customers_output_path}")

Saved cleaned customers data to: ..\data\processed\customers_clean.csv


In [20]:
# check the saved file
customers_check = pd.read_csv(
    customers_output_path,
    dtype={
        "customer_id": "string",
        "customer_unique_id": "string",
        "customer_zip_code_prefix": "string",
        "customer_city": "string",
        "customer_state": "string",
    },
)

customers_check.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [21]:
customers_check.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  string
 1   customer_unique_id        99441 non-null  string
 2   customer_zip_code_prefix  99441 non-null  string
 3   customer_city             99441 non-null  string
 4   customer_state            99441 non-null  string
dtypes: string(5)
memory usage: 3.8 MB


### Customers cleaning decisions

- Preserved all 99,441 source records.
- Converted `customer_zip_code_prefix` from numeric to a five-character string because it is an identifier, not a measurable value.
- Preserved leading zeros in ZIP-code prefixes.
- Removed surrounding whitespace from text columns.
- Standardized customer state codes to uppercase.
- No missing-value treatment was required.
- No exact duplicate rows or duplicate `customer_id` values were found.
- Repeated `customer_unique_id` values were retained because they represent customers who placed multiple orders.
- Saved the cleaned dataset as `data/processed/customers_clean.csv`.

# 2. Orders Dataset

In [22]:
orders_raw = raw_datasets["orders"]

orders_clean = orders_raw.copy()

In [23]:
orders_clean.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [24]:
orders_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [25]:
orders_clean.describe(include="all")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-03-31 15:08:21,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-14 20:02:44,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


In [27]:
# Convert datetime columns
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_columns:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col],
        errors="coerce"
    )

In [28]:
# Clean string columns
text_columns = [
    "order_id",
    "customer_id",
    "order_status",
]

for col in text_columns:
    orders_clean[col] = (
        orders_clean[col]
        .astype("string")
        .str.strip()
    )

In [29]:
# Standardizing order status column - lowercase
orders_clean["order_status"] = (
    orders_clean["order_status"]
    .str.lower()
)

In [30]:
orders_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  string        
 1   customer_id                    99441 non-null  string        
 2   order_status                   99441 non-null  string        
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), string(3)
memory usage: 6.1 MB


In [31]:
orders_clean.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

### Investigate missing order dates

In [ ]:
# Missing dates grouped by order status
missing_dates_by_status = (
    orders_clean
    .groupby("order_status")
    .agg(
        total_orders=("order_id", "count"),
        missing_approved_at=(
            "order_approved_at",
            lambda column: column.isna().sum(),
        ),
        missing_carrier_date=(
            "order_delivered_carrier_date",
            lambda column: column.isna().sum(),
        ),
        missing_customer_date=(
            "order_delivered_customer_date",
            lambda column: column.isna().sum(),
        ),
    )
    .sort_values("total_orders", ascending=False)
)

missing_dates_by_status

,total_orders,missing_approved_at,missing_carrier_date,missing_customer_date
order_status,,,,
delivered,96478,14,2,8
shipped,1107,0,0,1107
canceled,625,141,550,619
unavailable,609,0,609,609
invoiced,314,0,314,314
processing,301,0,301,301
created,5,5,5,5
approved,2,0,2,2


In [33]:
# Inspect delivered orders with missing delivery dates
delivered_with_missing_customer_date = orders_clean[
    (orders_clean["order_status"] == "delivered")
    & (orders_clean["order_delivered_customer_date"].isna())
]

delivered_with_missing_customer_date[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ]
].head(20)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [ ]:
# Total no of missing customer date, with missing date
len(delivered_with_missing_customer_date)

8

In [37]:
# Inspect delivered orders without carrier dates
delivered_with_missing_carrier_date = orders_clean[
    (orders_clean["order_status"] == "delivered")
    & (orders_clean["order_delivered_carrier_date"].isna())
]

delivered_with_missing_carrier_date

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23


In [36]:
len(delivered_with_missing_carrier_date)

2

In [38]:
# Inspect orders without approval timestamps
orders_without_approval = orders_clean[
    orders_clean["order_approved_at"].isna()
]

orders_without_approval["order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: Int64

In [39]:
orders_without_approval.head(20)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
1130,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,2018-08-28 15:26:39,NaT,NaT,NaT,2018-09-12
1801,ed3efbd3a87bea76c2812c66a0b32219,191984a8ba4cbb2145acb4fe35b69664,canceled,2018-09-20 13:54:16,NaT,NaT,NaT,2018-10-17
1868,df8282afe61008dc26c6c31011474d02,aa797b187b5466bc6925aaaa4bb3bed1,canceled,2017-03-04 12:14:30,NaT,NaT,NaT,2017-04-10
2029,8d4c637f1accf7a88a4555f02741e606,b1dd715db389a2077f43174e7a675d07,canceled,2018-08-29 16:27:49,NaT,NaT,NaT,2018-09-13
2161,7a9d4c7f9b068337875b95465330f2fc,7f71ae48074c0cfec9195f88fcbfac55,canceled,2017-05-01 16:12:39,NaT,NaT,NaT,2017-05-30
3056,ddaec6fff982b13e7e048b627a11d6da,68f4ad79cc0c2ad06e19088f5c00e9fa,canceled,2016-10-04 19:41:32,NaT,NaT,NaT,2016-11-16
3094,5290c34bd38a8a095b885f13958db1e1,92af427e290117f39d9ff908566072e0,canceled,2018-08-21 10:25:18,NaT,NaT,NaT,2018-09-06
3684,03310aa823a66056268a3bab36e827fb,25dbbf0c477fd4ae0880aaffbb12e8b3,canceled,2018-08-07 16:33:59,NaT,NaT,NaT,2018-09-04
4373,4c8b9947280829d0a8b7e81cc249b875,403c35c4d8813bf67b3d396b91ca1619,canceled,2018-08-09 14:54:47,NaT,NaT,NaT,2018-08-21
4939,b13ea375fe9c728832688264638f84cf,0dc5884bc5ffba283678229f27e07ff9,canceled,2018-08-22 18:52:29,NaT,NaT,NaT,2018-09-19


In [40]:
# Validate order IDs and duplicates
print("Exact duplicate rows:", orders_clean.duplicated().sum())
print(
    "Duplicate order_id:",
    orders_clean["order_id"].duplicated().sum(),
)
print(
    "Duplicate customer_id:",
    orders_clean["customer_id"].duplicated().sum(),
)

Exact duplicate rows: 0
Duplicate order_id: 0
Duplicate customer_id: 0


In [41]:
# Check invalid chronological sequences
approved_before_purchase = orders_clean[
    orders_clean["order_approved_at"]
    < orders_clean["order_purchase_timestamp"]
]

carrier_before_approval = orders_clean[
    orders_clean["order_delivered_carrier_date"]
    < orders_clean["order_approved_at"]
]

customer_before_carrier = orders_clean[
    orders_clean["order_delivered_customer_date"]
    < orders_clean["order_delivered_carrier_date"]
]

print(
    "Approval before purchase:",
    len(approved_before_purchase),
)

print(
    "Carrier delivery before approval:",
    len(carrier_before_approval),
)

print(
    "Customer delivery before carrier delivery:",
    len(customer_before_carrier),
)

Approval before purchase: 0
Carrier delivery before approval: 1359
Customer delivery before carrier delivery: 23


### Key analysis :
- Missing order dates are mostly explained by the order status.
- Shipped orders correctly have no customer delivery date because delivery was not yet recorded.
- Cancelled, unavailable, invoiced, processing, created and approved orders commonly lack carrier or customer delivery dates.
- Eight delivered orders have no recorded customer delivery date.
- Two delivered orders have no recorded carrier date.
- Fourteen delivered orders have no approval timestamp.
- These incomplete delivered records will be retained because their missing values cannot be reconstructed reliably.
- There are no exact duplicate rows or duplicate `order_id` values.
- No approval timestamp occurs before the purchase timestamp.
- There are 1,359 records where the carrier timestamp is earlier than the approval timestamp.
- There are 23 records where the customer delivery timestamp is earlier than the carrier timestamp.
- These timestamp inconsistencies will be preserved and documented rather than overwritten without a reliable correction rule.
- Missing datetime values will remain as `NaT`.

In [42]:
# Final validation
print("Original shape:", orders_raw.shape)
print("Cleaned shape:", orders_clean.shape)
print("Unique order IDs:", orders_clean["order_id"].nunique())
print("Exact duplicate rows:", orders_clean.duplicated().sum())
print("\nMissing values:")
print(orders_clean.isna().sum())

Original shape: (99441, 8)
Cleaned shape: (99441, 8)
Unique order IDs: 99441
Exact duplicate rows: 0

Missing values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [43]:
# Saving the dataset
orders_output_path = PROCESSED_DATA_DIR / "orders_clean.csv"

orders_clean.to_csv(
    orders_output_path,
    index=False,
)

print(f"Saved cleaned orders data to: {orders_output_path}")

Saved cleaned orders data to: ..\data\processed\orders_clean.csv


In [44]:
# checking
orders_check = pd.read_csv(
    orders_output_path,
    dtype={
        "order_id": "string",
        "customer_id": "string",
        "order_status": "string",
    },
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)

orders_check.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  string        
 1   customer_id                    99441 non-null  string        
 2   order_status                   99441 non-null  string        
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), string(3)
memory usage: 6.1 MB


### Orders cleaning decisions

- Converted all five timestamp columns to datetime.
- Standardized identifier and order-status columns as strings.
- Retained all source records.
- Preserved missing timestamps because they are generally explained by order status and cannot be reconstructed reliably.
- Retained timestamp inconsistencies for later data-quality reporting instead of modifying source values.
- No duplicate rows or duplicate order IDs were removed.
- No analytical or derived delivery columns were created during cleaning.
- Saved the cleaned dataset as `data/processed/orders_clean.csv`.

# Order Items dataset

In [45]:
order_items_raw = raw_datasets["order_items"]
order_items_clean = order_items_raw.copy()

# Convert text columns
text_columns = [
    "order_id",
    "product_id",
    "seller_id",
]

for column in text_columns:
    order_items_clean[column] = (
        order_items_clean[column]
        .astype("string")
        .str.strip()
    )

# Convert timestamp
order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"],
    errors="coerce",
)

In [46]:
# Validation
print(order_items_clean.info())
print("\nMissing values:")
print(order_items_clean.isna().sum())
print("\nExact duplicates:", order_items_clean.duplicated().sum())
print(
    "Duplicate composite key:",
    order_items_clean.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum(),
)
print("\nOriginal shape:", order_items_raw.shape)
print("Cleaned shape:", order_items_clean.shape)

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  string        
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  string        
 3   seller_id            112650 non-null  string        
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), string(3)
memory usage: 6.0 MB
None

Missing values:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Exact duplicates: 0
Duplicate composite key: 0

Original shape: (112650, 7)
Cleaned shap

In [48]:
order_items_clean.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [47]:
print("Non-positive prices:", (order_items_clean["price"] <= 0).sum())
print("Negative freight values:", (order_items_clean["freight_value"] < 0).sum())
print("Invalid item IDs:", (order_items_clean["order_item_id"] < 1).sum())

Non-positive prices: 0
Negative freight values: 0
Invalid item IDs: 0


### Order Items cleaning decisions

- Converted `order_id`, `product_id` and `seller_id` to string datatype.
- Converted `shipping_limit_date` to datetime.
- No missing values were found.
- No exact duplicate rows were found.
- No duplicate (`order_id`, `order_item_id`) combinations were found.
- No non-positive prices were found.
- No negative freight values were found.
- All `order_item_id` values are valid.
- Dataset required only datatype standardization.

In [49]:
# Save cleaned order items dataset

order_items_output_path = (
    PROCESSED_DATA_DIR / "order_items_clean.csv"
)

order_items_clean.to_csv(
    order_items_output_path,
    index=False,
)

print(f"Saved cleaned order items data to: {order_items_output_path}")

Saved cleaned order items data to: ..\data\processed\order_items_clean.csv


In [50]:
order_items_check = pd.read_csv(
    order_items_output_path,
    dtype={
        "order_id": "string",
        "product_id": "string",
        "seller_id": "string",
    },
    parse_dates=["shipping_limit_date"],
)

order_items_check.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  string        
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  string        
 3   seller_id            112650 non-null  string        
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), string(3)
memory usage: 6.0 MB


# Products dataset

In [8]:
products_raw = pd.read_csv(
    RAW_DATA_DIR / "olist_products_dataset.csv"
)

products_raw.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [9]:
# copy to new
products_clean = products_raw.copy()

In [10]:
# Basic infos
products_clean.info()

print("\nMissing values:")
print(products_clean.isnull().sum())

print("\nExact duplicates:")
print(products_clean.duplicated().sum())

print("\nOriginal shape:", products_raw.shape)

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB

Missing values:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product

In [11]:
products_clean["product_id"] = (
    products_clean["product_id"]
    .astype("string")
)

In [12]:
# Validation
print("Duplicate product IDs:",
      products_clean["product_id"].duplicated().sum())

print("Negative weights:",
      (products_clean["product_weight_g"] < 0).sum())

print("Negative name length:",
      (products_clean["product_name_lenght"] < 0).sum())

print("Negative description length:",
      (products_clean["product_description_lenght"] < 0).sum())

print("Negative photo quantity:",
      (products_clean["product_photos_qty"] < 0).sum())

Duplicate product IDs: 0
Negative weights: 0
Negative name length: 0
Negative description length: 0
Negative photo quantity: 0


### Cleaning Decisions

- Converted `product_id` to pandas StringDtype.
- No duplicate records found.
- No invalid numeric values detected.
- Kept missing product metadata (610 products) because deleting these products could break joins with order_items.
- Kept missing physical dimensions (2 products) because these represent unavailable product information rather than data corruption.
- No rows removed.

In [13]:
# Renaming the column names
products_clean = products_clean.rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length",
        
    }
)

In [14]:
products_output_path = (
    PROCESSED_DATA_DIR / "products_clean.csv"
)

products_clean.to_csv(
    products_output_path,
    index=False
)

print(f"Saved cleaned products data to: {products_output_path}")

Saved cleaned products data to: ..\data\processed\products_clean.csv


In [15]:
products_check = pd.read_csv(
    products_output_path,
    dtype={
        "product_id": "string",
        "product_category_name": "string",
    }
)

products_check.info()
products_check.head()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  string 
 1   product_category_name       32341 non-null  string 
 2   product_name_length         32341 non-null  float64
 3   product_description_length  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), string(2)
memory usage: 2.3 MB


,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


# Sellers dataset

In [58]:
sellers_raw = pd.read_csv(
    RAW_DATA_DIR / "olist_sellers_dataset.csv"
)

sellers_raw.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [59]:
sellers_clean = sellers_raw.copy()

In [60]:
# Basic info
sellers_clean.info()

print("\nMissing values:")
print(sellers_clean.isnull().sum())

print("\nExact duplicates:")
print(sellers_clean.duplicated().sum())

print("\nOriginal shape:", sellers_raw.shape)

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB

Missing values:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

Exact duplicates:
0

Original shape: (3095, 4)


In [61]:
# Treat IDs and ZIP codes as strings.
sellers_clean["seller_id"] = (
    sellers_clean["seller_id"]
    .astype("string")
)

sellers_clean["seller_zip_code_prefix"] = (
    sellers_clean["seller_zip_code_prefix"]
    .astype("string")
    .str.zfill(5)
)

sellers_clean["seller_city"] = (
    sellers_clean["seller_city"]
    .astype("string")
)

sellers_clean["seller_state"] = (
    sellers_clean["seller_state"]
    .astype("string")
)

In [62]:
# Validation
print(
    "Duplicate seller IDs:",
    sellers_clean["seller_id"].duplicated().sum()
)

print(
    "Invalid ZIP length:",
    (sellers_clean["seller_zip_code_prefix"].str.len() != 5).sum()
)

print(
    "Missing seller IDs:",
    sellers_clean["seller_id"].isna().sum()
)

Duplicate seller IDs: 0
Invalid ZIP length: 0
Missing seller IDs: 0


### Cleaning Decisions

- Converted `seller_id` to pandas StringDtype.
- Converted `seller_zip_code_prefix` to a 5-character string using leading zeros where necessary.
- Converted `seller_city` and `seller_state` to StringDtype.
- No missing values found.
- No duplicate records found.
- No duplicate seller IDs found.
- No rows removed.

In [63]:
# save dataset
sellers_output_path = (
    PROCESSED_DATA_DIR / "sellers_clean.csv"
)

sellers_clean.to_csv(
    sellers_output_path,
    index=False
)

print(f"Saved cleaned sellers data to: {sellers_output_path}")

Saved cleaned sellers data to: ..\data\processed\sellers_clean.csv


In [64]:
sellers_check = pd.read_csv(
    sellers_output_path,
    dtype={
        "seller_id": "string",
        "seller_zip_code_prefix": "string",
        "seller_city": "string",
        "seller_state": "string"
    }
)

sellers_check.info()
sellers_check.head()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   string
 1   seller_zip_code_prefix  3095 non-null   string
 2   seller_city             3095 non-null   string
 3   seller_state            3095 non-null   string
dtypes: string(4)
memory usage: 96.8 KB


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


# Payments dataset

In [65]:
payments_raw = pd.read_csv(
    RAW_DATA_DIR / "olist_order_payments_dataset.csv"
)

payments_raw.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [66]:
payments_clean = payments_raw.copy()

In [67]:
payments_clean.info()

print("\nMissing values:")
print(payments_clean.isnull().sum())

print("\nExact duplicates:")
print(payments_clean.duplicated().sum())

print("\nOriginal shape:", payments_raw.shape)

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB

Missing values:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

Exact duplicates:
0

Original shape: (103886, 5)


In [68]:
# convert data types
payments_clean["order_id"] = (
    payments_clean["order_id"]
    .astype("string")
)

payments_clean["payment_type"] = (
    payments_clean["payment_type"]
    .astype("string")
)

In [69]:
# validation
print(
    "Negative payment values:",
    (payments_clean["payment_value"] < 0).sum()
)

print(
    "Invalid installments:",
    (payments_clean["payment_installments"] < 0).sum()
)

print(
    "Duplicate rows:",
    payments_clean.duplicated().sum()
)

print(
    "Missing order IDs:",
    payments_clean["order_id"].isna().sum()
)

Negative payment values: 0
Invalid installments: 0
Duplicate rows: 0
Missing order IDs: 0


### Cleaning Decisions

- Converted `order_id` and `payment_type` to pandas StringDtype.
- No missing values found.
- No duplicate records found.
- No invalid payment amounts detected.
- No invalid installment values detected.
- No rows removed.

In [70]:
payments_output_path = (
    PROCESSED_DATA_DIR / "payments_clean.csv"
)

payments_clean.to_csv(
    payments_output_path,
    index=False
)

print(f"Saved cleaned payments data to: {payments_output_path}")

Saved cleaned payments data to: ..\data\processed\payments_clean.csv


In [71]:
payments_check = pd.read_csv(
    payments_output_path,
    dtype={
        "order_id": "string",
        "payment_type": "string"
    }
)

payments_check.info()
payments_check.head()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  string 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  string 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), string(2)
memory usage: 4.0 MB


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


# Reviews dataset

In [72]:
reviews_raw = pd.read_csv(
    RAW_DATA_DIR / "olist_order_reviews_dataset.csv"
)

reviews_raw.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Pá...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [73]:
reviews_clean = reviews_raw.copy()

In [74]:
reviews_clean.info()

print("\nMissing values:")
print(reviews_clean.isnull().sum())

print("\nExact duplicates:")
print(reviews_clean.duplicated().sum())

print("\nOriginal shape:", reviews_raw.shape)

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB

Missing values:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

Exact duplicates:
0

Original shape: (99224, 7)


In [75]:
# Converting datatypes
reviews_clean["review_id"] = (
    reviews_clean["review_id"]
    .astype("string")
)

reviews_clean["order_id"] = (
    reviews_clean["order_id"]
    .astype("string")
)

reviews_clean["review_comment_title"] = (
    reviews_clean["review_comment_title"]
    .astype("string")
)

reviews_clean["review_comment_message"] = (
    reviews_clean["review_comment_message"]
    .astype("string")
)

date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

reviews_clean[date_columns] = (
    reviews_clean[date_columns]
    .apply(pd.to_datetime)
)

In [76]:
# validations
print(
    "Duplicate review IDs:",
    reviews_clean["review_id"].duplicated().sum()
)

print(
    "Negative review scores:",
    (reviews_clean["review_score"] < 1).sum()
)

print(
    "Review scores > 5:",
    (reviews_clean["review_score"] > 5).sum()
)

print(
    "Missing review IDs:",
    reviews_clean["review_id"].isna().sum()
)

Duplicate review IDs: 814
Negative review scores: 0
Review scores > 5: 0
Missing review IDs: 0


In [78]:
## Investingating duplicate values
duplicate_reviews = reviews_clean[
    reviews_clean["review_id"].duplicated(keep=False)
].sort_values("review_id")

duplicate_reviews.head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,<NA>,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequentemente não funcionou....",2018-03-07,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,<NA>,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequentemente não funcionou....",2018-03-07,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,<NA>,<NA>,2017-09-21,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,<NA>,<NA>,2017-09-21,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,<NA>,Produto entregue dentro de embalagem do fornecedor sem os parafusos de fixação das partes.,2018-03-07,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,<NA>,Produto entregue dentro de embalagem do fornecedor sem os parafusos de fixação das partes.,2018-03-07,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,<NA>,<NA>,2018-03-02,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,<NA>,<NA>,2018-03-02,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,<NA>,"O pedido consta de 2 produtos e até agora recebi apenas 1 produto, e o que me preocupa é que o s...",2017-09-09,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,<NA>,"O pedido consta de 2 produtos e até agora recebi apenas 1 produto, e o que me preocupa é que o s...",2017-09-09,2017-09-13 09:52:44


A review can be associated with multiple orders (for example, when a customer purchases multiple products and the platform links the same review to more than one order).

If we remove one of these rows:

we would lose the relationship between that review and one of the orders,
which would break referential integrity when joining with the orders table.

So although review_id is not unique, each (review_id, order_id) pair is unique, which is what matters.

In [ ]:
# cheking any duplicate rows
duplicate_reviews.duplicated().sum()

np.int64(0)

In [80]:
# unique reviews
duplicate_reviews["review_id"].nunique()

789

In [81]:
# Cheking one record
duplicate_reviews[
    duplicate_reviews["review_id"] ==
    duplicate_reviews["review_id"].iloc[0]
]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,<NA>,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequentemente não funcionou....",2018-03-07,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,<NA>,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequentemente não funcionou....",2018-03-07,2018-03-20 18:08:23


### Cleaning Decisions

- Converted text columns to pandas StringDtype.
- Converted review dates to datetime.
- Kept missing review titles and messages because many customers only provided a review score.
- Kept duplicated `review_id`s because they are associated with different `order_id`s and represent valid relationships.
- No duplicate rows found.
- No invalid review scores detected.
- No rows removed.

In [82]:
reviews_output_path = (
    PROCESSED_DATA_DIR / "reviews_clean.csv"
)

reviews_clean.to_csv(
    reviews_output_path,
    index=False
)

print(f"Saved cleaned reviews data to: {reviews_output_path}")

Saved cleaned reviews data to: ..\data\processed\reviews_clean.csv


In [83]:
reviews_check = pd.read_csv(
    reviews_output_path,
    dtype={
        "review_id": "string",
        "order_id": "string",
        "review_comment_title": "string",
        "review_comment_message": "string"
    },
    parse_dates=[
        "review_creation_date",
        "review_answer_timestamp"
    ]
)

reviews_check.info()
reviews_check.head()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  string        
 1   order_id                 99224 non-null  string        
 2   review_score             99224 non-null  int64         
 3   review_comment_title     11568 non-null  string        
 4   review_comment_message   40977 non-null  string        
 5   review_creation_date     99224 non-null  datetime64[us]
 6   review_answer_timestamp  99224 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), string(4)
memory usage: 5.3 MB


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,<NA>,<NA>,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,<NA>,<NA>,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,<NA>,<NA>,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,<NA>,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,<NA>,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Pá...,2018-03-01,2018-03-02 10:26:53


# Geolocation dataset

In [84]:
geolocation_raw = raw_datasets["geolocation"]
geolocation_clean = geolocation_raw.copy()


In [87]:
geolocation_clean.tail(2)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1000161,99980,-28.388932,-51.846871,david canabarro,RS
1000162,99950,-28.070104,-52.018658,tapejara,RS


In [86]:
geolocation_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 38.2 MB


In [88]:
# Convert ZIP prefix to a 5-character string
geolocation_clean["geolocation_zip_code_prefix"] = (
    geolocation_clean["geolocation_zip_code_prefix"]
    .astype("string")
    .str.zfill(5)
)


In [89]:
# Convert text columns
geolocation_clean["geolocation_city"] = (
    geolocation_clean["geolocation_city"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [90]:
geolocation_clean["geolocation_state"] = (
    geolocation_clean["geolocation_state"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [91]:
geolocation_clean.duplicated().sum()

np.int64(261831)

In [92]:
# Remove only exact duplicate rows
geolocation_clean = (
    geolocation_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

In [93]:
# Validation
print(geolocation_clean.info())

print("\nMissing values:")
print(geolocation_clean.isna().sum())

print("\nOriginal shape:", geolocation_raw.shape)
print("Cleaned shape:", geolocation_clean.shape)

<class 'pandas.DataFrame'>
RangeIndex: 738332 entries, 0 to 738331
Data columns (total 5 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   geolocation_zip_code_prefix  738332 non-null  string 
 1   geolocation_lat              738332 non-null  float64
 2   geolocation_lng              738332 non-null  float64
 3   geolocation_city             738332 non-null  string 
 4   geolocation_state            738332 non-null  string 
dtypes: float64(2), string(3)
memory usage: 28.2 MB
None

Missing values:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

Original shape: (1000163, 5)
Cleaned shape: (738332, 5)


In [94]:
print(
    "Remaining exact duplicates:",
    geolocation_clean.duplicated().sum(),
)

print(
    "Invalid ZIP lengths:",
    (
        geolocation_clean[
            "geolocation_zip_code_prefix"
        ].str.len() != 5
    ).sum(),
)


Remaining exact duplicates: 0
Invalid ZIP lengths: 0


In [95]:
print(
    "Invalid latitude values:",
    (
        (geolocation_clean["geolocation_lat"] < -90)
        | (geolocation_clean["geolocation_lat"] > 90)
    ).sum(),
)

print(
    "Invalid longitude values:",
    (
        (geolocation_clean["geolocation_lng"] < -180)
        | (geolocation_clean["geolocation_lng"] > 180)
    ).sum(),
)

Invalid latitude values: 0
Invalid longitude values: 0


### Cleaning Decisions

- Converted `geolocation_zip_code_prefix` to a 5-character string.
- Standardized city names to lowercase and state codes to uppercase.
- Removed exact duplicate rows.
- Retained multiple coordinate records for the same ZIP-code prefix because they are valid source observations.
- Did not aggregate coordinates during the cleaning phase.
- No missing values were found.

In [96]:
# save
geolocation_output_path = (
    PROCESSED_DATA_DIR / "geolocation_clean.csv"
)

geolocation_clean.to_csv(
    geolocation_output_path,
    index=False,
)

print(
    f"Saved cleaned geolocation data to: "
    f"{geolocation_output_path}"
)

Saved cleaned geolocation data to: ..\data\processed\geolocation_clean.csv


In [97]:
geolocation_check = pd.read_csv(
    geolocation_output_path,
    dtype={
        "geolocation_zip_code_prefix": "string",
        "geolocation_city": "string",
        "geolocation_state": "string",
    },
)

geolocation_check.info()
geolocation_check.head()

<class 'pandas.DataFrame'>
RangeIndex: 738332 entries, 0 to 738331
Data columns (total 5 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   geolocation_zip_code_prefix  738332 non-null  string 
 1   geolocation_lat              738332 non-null  float64
 2   geolocation_lng              738332 non-null  float64
 3   geolocation_city             738332 non-null  string 
 4   geolocation_state            738332 non-null  string 
dtypes: float64(2), string(3)
memory usage: 28.2 MB


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP
3,01041,-23.544392,-46.639499,sao paulo,SP
4,01035,-23.541578,-46.641607,sao paulo,SP


### Geolocation cleaning results

- Removed 261,831 exact duplicate rows.
- Reduced the dataset from 1,000,163 rows to 738,332 rows.
- Preserved all non-duplicate coordinate records.
- Converted ZIP-code prefixes to five-character strings.
- Standardized city names and state codes.
- No missing values remain.
- No invalid ZIP lengths were found.
- All latitude values are within `-90` to `90`.
- All longitude values are within `-180` to `180`.

# Category Translation Dataset

In [98]:
category_translation_raw = raw_datasets["category_translation"]
category_translation_clean = category_translation_raw.copy()

In [99]:
# Convert text columns
category_translation_clean["product_category_name"] = (
    category_translation_clean["product_category_name"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [100]:
category_translation_clean["product_category_name_english"] = (
    category_translation_clean["product_category_name_english"]
    .astype("string")
    .str.strip()
    .str.lower()
)


In [101]:
category_translation_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     string
 1   product_category_name_english  71 non-null     string
dtypes: string(2)
memory usage: 1.2 KB


In [102]:
# Validation
category_translation_clean.info()

print("\nMissing values:")
print(category_translation_clean.isna().sum())

print("\nExact duplicates:")
print(category_translation_clean.duplicated().sum())


<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     string
 1   product_category_name_english  71 non-null     string
dtypes: string(2)
memory usage: 1.2 KB

Missing values:
product_category_name            0
product_category_name_english    0
dtype: int64

Exact duplicates:
0


In [103]:
print(
    "\nDuplicate Portuguese category names:",
    category_translation_clean[
        "product_category_name"
    ].duplicated().sum(),
)

print(
    "Duplicate English category names:",
    category_translation_clean[
        "product_category_name_english"
    ].duplicated().sum(),
)

print("\nOriginal shape:", category_translation_raw.shape)
print("Cleaned shape:", category_translation_clean.shape)


Duplicate Portuguese category names: 0
Duplicate English category names: 0

Original shape: (71, 2)
Cleaned shape: (71, 2)


### Cleaning Decisions

- Converted both category-name columns to pandas StringDtype.
- Removed surrounding whitespace.
- Standardized category names to lowercase.
- No missing values were found.
- No duplicate rows were found.
- Both Portuguese and English category names remain unique.
- No rows were removed.

In [104]:
category_translation_output_path = (
    PROCESSED_DATA_DIR / "category_translation_clean.csv"
)

category_translation_clean.to_csv(
    category_translation_output_path,
    index=False,
)

print(
    "Saved cleaned category translation data to:",
    category_translation_output_path,
)

Saved cleaned category translation data to: ..\data\processed\category_translation_clean.csv


In [105]:
category_translation_check = pd.read_csv(
    category_translation_output_path,
    dtype={
        "product_category_name": "string",
        "product_category_name_english": "string",
    },
)

category_translation_check.info()
category_translation_check.head()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     string
 1   product_category_name_english  71 non-null     string
dtypes: string(2)
memory usage: 1.2 KB


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
